# ETF Sector and Country Exposure

This notebook calculates sector and country concentration risk using the percentage allocation of each ETF. Exposure data will be stored in long format so each ETF can have multiple sector and country rows.

In [1]:
import pandas as pd

sector_template = pd.DataFrame(
    columns=["Ticker", "Sector", "Weight_Percent"]
)

country_template = pd.DataFrame(
    columns=["Ticker", "Country", "Weight_Percent"]
)

sector_template.to_csv(
    "../data/raw/etf_sector_exposure.csv",
    index=False
)

country_template.to_csv(
    "../data/raw/etf_country_exposure.csv",
    index=False
)

print("Sector and country exposure templates created.")

Sector and country exposure templates created.


In [2]:
etf_universe = pd.read_csv(
    "../data/processed/etf_universe_clean.csv"
)

print("Asset classes:")
print(etf_universe["Asset_Class"].value_counts())

print("\nETF list:")
display(
    etf_universe[
        [
            "ETF_ID",
            "Ticker",
            "ETF_Name",
            "Asset_Class",
            "Sector_Focus"
        ]
    ]
)

Asset classes:
Asset_Class
Equity      12
Bond         6
Gold         1
Property     1
Name: count, dtype: int64

ETF list:


,ETF_ID,Ticker,ETF_Name,Asset_Class,Sector_Focus
0,VWRP,VWRP.L,Vanguard FTSE All-World UCITS ETF,Equity,Broad
1,SWDA,SWDA.L,iShares Core MSCI World UCITS ETF,Equity,Broad
2,VUAG,VUAG.L,Vanguard S&P 500 UCITS ETF,Equity,Broad large-cap
3,ISF,ISF.L,iShares Core FTSE 100 UCITS ETF,Equity,Broad large-cap
4,EQQQ,EQQQ.L,Invesco EQQQ Nasdaq-100 UCITS ETF,Equity,Growth / Nasdaq
5,VEUR,VEUR.L,Vanguard FTSE Developed Europe UCITS ETF,Equity,Broad
6,VJPN,VJPN.L,Vanguard FTSE Japan UCITS ETF,Equity,Broad
7,VFEG,VFEG.L,Vanguard FTSE Emerging Markets UCITS ETF,Equity,Broad
8,ICHN,ICHN.AS,iShares MSCI China UCITS ETF,Equity,Regional
9,IUIT,IUIT.L,iShares S&P 500 Information Technology Sector ...,Equity,Technology


In [3]:
exposure_requirements = etf_universe[
    [
        "ETF_ID",
        "Ticker",
        "ETF_Name",
        "Asset_Class"
    ]
].copy()

exposure_requirements["Needs_Sector_Data"] = (
    exposure_requirements["Asset_Class"]
    .astype(str)
    .str.contains("Equity", case=False, na=False)
)

exposure_requirements["Needs_Country_Data"] = (
    ~exposure_requirements["Asset_Class"]
    .astype(str)
    .str.contains(
        "Alternative|Commodity|Gold",
        case=False,
        na=False
    )
)

display(exposure_requirements)

print(
    "ETFs requiring sector data:",
    exposure_requirements["Needs_Sector_Data"].sum()
)

print(
    "ETFs requiring country data:",
    exposure_requirements["Needs_Country_Data"].sum()
)

,ETF_ID,Ticker,ETF_Name,Asset_Class,Needs_Sector_Data,Needs_Country_Data
0,VWRP,VWRP.L,Vanguard FTSE All-World UCITS ETF,Equity,True,True
1,SWDA,SWDA.L,iShares Core MSCI World UCITS ETF,Equity,True,True
2,VUAG,VUAG.L,Vanguard S&P 500 UCITS ETF,Equity,True,True
3,ISF,ISF.L,iShares Core FTSE 100 UCITS ETF,Equity,True,True
4,EQQQ,EQQQ.L,Invesco EQQQ Nasdaq-100 UCITS ETF,Equity,True,True
5,VEUR,VEUR.L,Vanguard FTSE Developed Europe UCITS ETF,Equity,True,True
6,VJPN,VJPN.L,Vanguard FTSE Japan UCITS ETF,Equity,True,True
7,VFEG,VFEG.L,Vanguard FTSE Emerging Markets UCITS ETF,Equity,True,True
8,ICHN,ICHN.AS,iShares MSCI China UCITS ETF,Equity,True,True
9,IUIT,IUIT.L,iShares S&P 500 Information Technology Sector ...,Equity,True,True


ETFs requiring sector data: 12
ETFs requiring country data: 19


In [4]:
print("No sector data required:")
display(
    exposure_requirements.loc[
        ~exposure_requirements["Needs_Sector_Data"],
        ["Ticker", "ETF_Name", "Asset_Class"]
    ]
)

print("\nNo country data required:")
display(
    exposure_requirements.loc[
        ~exposure_requirements["Needs_Country_Data"],
        ["Ticker", "ETF_Name", "Asset_Class"]
    ]
)

No sector data required:


,Ticker,ETF_Name,Asset_Class
12,VGOV.L,Vanguard U.K. Gilt UCITS ETF,Bond
13,IGLO.L,iShares Global Govt Bond UCITS ETF,Bond
14,IBTG.L,iShares $ Treasury Bond 1-3yr UCITS ETF GBP He...,Bond
15,AGGG.L,iShares Core Global Aggregate Bond UCITS ETF,Bond
16,CORP.L,iShares Global Corp Bond UCITS ETF,Bond
17,GHYS.L,iShares Global High Yield Corp Bond GBP Hedged...,Bond
18,SGLN.L,iShares Physical Gold ETC,Gold
19,IWDP.L,iShares Developed Markets Property Yield UCITS...,Property



No country data required:


,Ticker,ETF_Name,Asset_Class
18,SGLN.L,iShares Physical Gold ETC,Gold


In [5]:
exposure_requirements.loc[
    ~exposure_requirements["Needs_Country_Data"],
    ["Ticker", "ETF_Name", "Asset_Class"]
]

,Ticker,ETF_Name,Asset_Class
18,SGLN.L,iShares Physical Gold ETC,Gold


In [6]:
sector_etfs = exposure_requirements.loc[
    exposure_requirements["Needs_Sector_Data"],
    ["ETF_ID", "Ticker", "ETF_Name", "Asset_Class"]
].copy()

country_etfs = exposure_requirements.loc[
    exposure_requirements["Needs_Country_Data"],
    ["ETF_ID", "Ticker", "ETF_Name", "Asset_Class"]
].copy()

sector_etfs.to_csv(
    "../data/processed/sector_data_requirements.csv",
    index=False
)

country_etfs.to_csv(
    "../data/processed/country_data_requirements.csv",
    index=False
)

print("Sector ETFs:", len(sector_etfs))
print("Country ETFs:", len(country_etfs))

display(sector_etfs)

Sector ETFs: 12
Country ETFs: 19


,ETF_ID,Ticker,ETF_Name,Asset_Class
0,VWRP,VWRP.L,Vanguard FTSE All-World UCITS ETF,Equity
1,SWDA,SWDA.L,iShares Core MSCI World UCITS ETF,Equity
2,VUAG,VUAG.L,Vanguard S&P 500 UCITS ETF,Equity
3,ISF,ISF.L,iShares Core FTSE 100 UCITS ETF,Equity
4,EQQQ,EQQQ.L,Invesco EQQQ Nasdaq-100 UCITS ETF,Equity
5,VEUR,VEUR.L,Vanguard FTSE Developed Europe UCITS ETF,Equity
6,VJPN,VJPN.L,Vanguard FTSE Japan UCITS ETF,Equity
7,VFEG,VFEG.L,Vanguard FTSE Emerging Markets UCITS ETF,Equity
8,ICHN,ICHN.AS,iShares MSCI China UCITS ETF,Equity
9,IUIT,IUIT.L,iShares S&P 500 Information Technology Sector ...,Equity


In [7]:
sector_template = pd.DataFrame(
    columns=[
        "Ticker",
        "Sector",
        "Weight_Percent",
        "Data_As_Of",
        "Source_URL"
    ]
)

country_template = pd.DataFrame(
    columns=[
        "Ticker",
        "Country",
        "Weight_Percent",
        "Data_As_Of",
        "Source_URL"
    ]
)

sector_template.to_csv(
    "../data/raw/etf_sector_exposure.csv",
    index=False
)

country_template.to_csv(
    "../data/raw/etf_country_exposure.csv",
    index=False
)

print("Updated exposure templates created.")

Updated exposure templates created.


In [8]:
import pandas as pd

workbook_path = "../data/raw/ETF_Risk_Exposure_Workbook.xlsx"

sector_data = pd.read_excel(
    workbook_path,
    sheet_name="Sector_Exposure",
    header=3
)

country_data = pd.read_excel(
    workbook_path,
    sheet_name="Country_Exposure",
    header=3
)

print("Sector rows:", sector_data.shape[0])
print("Country rows:", country_data.shape[0])

display(sector_data.head())
display(country_data.head())

Sector rows: 105
Country rows: 90


,ETF_ID,ETF_Name,Ticker,Asset_Class,Sector,Weight (%),Source_Quality,Data_As_Of,Source_URL,Notes
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Technology,35.1,Exact provider,2026-06-30,https://fund-docs.vanguard.com/FTSE_All-World_...,Official Vanguard factsheet sector allocation.
1,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Financials,14.8,Exact provider,2026-06-30,https://fund-docs.vanguard.com/FTSE_All-World_...,Official Vanguard factsheet sector allocation.
2,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Industrials,12.6,Exact provider,2026-06-30,https://fund-docs.vanguard.com/FTSE_All-World_...,Official Vanguard factsheet sector allocation.
3,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Consumer Discretionary,11.0,Exact provider,2026-06-30,https://fund-docs.vanguard.com/FTSE_All-World_...,Official Vanguard factsheet sector allocation.
4,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Health Care,7.8,Exact provider,2026-06-30,https://fund-docs.vanguard.com/FTSE_All-World_...,Official Vanguard factsheet sector allocation.


,ETF_ID,ETF_Name,Ticker,Asset_Class,Country / Geography,Weight (%),Source_Quality,Data_As_Of,Source_URL,Notes
0,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,United States,61.70,Exact provider,2026-06-30,https://www.vanguard.co.uk/professional/produc...,Official Vanguard market allocation.
1,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Japan,5.90,Exact provider,2026-06-30,https://www.vanguard.co.uk/professional/produc...,Official Vanguard market allocation.
2,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Taiwan,3.38,Exact provider,2026-06-30,https://www.vanguard.co.uk/professional/produc...,Official Vanguard market allocation.
3,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,United Kingdom,3.18,Exact provider,2026-06-30,https://www.vanguard.co.uk/professional/produc...,Official Vanguard market allocation.
4,VWRP,Vanguard FTSE All-World UCITS ETF,VWRP.L,Equity,Canada,2.91,Exact provider,2026-06-30,https://www.vanguard.co.uk/professional/produc...,Official Vanguard market allocation.


In [9]:
sector_data["Weight (%)"] = pd.to_numeric(
    sector_data["Weight (%)"],
    errors="coerce"
)

country_data["Weight (%)"] = pd.to_numeric(
    country_data["Weight (%)"],
    errors="coerce"
)

sector_totals = (
    sector_data.groupby("Ticker")["Weight (%)"]
    .sum()
    .round(2)
)

country_totals = (
    country_data.groupby("Ticker")["Weight (%)"]
    .sum()
    .round(2)
)

print("Sector weight totals:")
display(sector_totals)

print("\nCountry weight totals:")
display(country_totals)

Sector weight totals:


Ticker
EQQQ.L     100.00
ICHN.AS    100.00
INRG.L     100.00
ISF.L      100.01
IUHC.L     100.00
IUIT.L     100.00
SWDA.L     100.02
VEUR.L     100.00
VFEG.L     100.00
VJPN.L     100.00
VUAG.L     100.00
VWRP.L     100.00
Name: Weight (%), dtype: float64


Country weight totals:


Ticker
AGGG.L     100.00
CORP.L     100.00
EQQQ.L     100.00
GHYS.L     100.00
IBTG.L     100.00
ICHN.AS    100.00
IGLO.L     100.00
INRG.L     100.00
ISF.L      100.00
IUHC.L     100.00
IUIT.L     100.00
IWDP.L     100.00
SWDA.L      99.99
VEUR.L     100.04
VFEG.L      98.00
VGOV.L     100.00
VJPN.L     100.00
VUAG.L     100.00
VWRP.L     100.00
Name: Weight (%), dtype: float64

In [10]:
vfeq_other_mask = (
    (country_data["Ticker"] == "VFEG.L")
    & (country_data["Country / Geography"] == "Other")
)

country_data.loc[vfeq_other_mask, "Weight (%)"] = 2.18

country_totals = (
    country_data.groupby("Ticker")["Weight (%)"]
    .sum()
    .round(2)
)

print("VFEG.L total:", country_totals.loc["VFEG.L"])

VFEG.L total: 100.0


In [11]:
def calculate_concentration(data):
    concentration = (
        data.groupby("Ticker")
        .agg(
            Total_Weight=("Weight (%)", "sum"),
            Largest_Exposure=("Weight (%)", "max"),
            Number_of_Categories=("Weight (%)", "count")
        )
    )

    hhi = data.groupby("Ticker")["Weight (%)"].apply(
        lambda weights: ((weights / 100) ** 2).sum()
    )

    concentration["HHI"] = hhi
    concentration["Effective_Categories"] = 1 / concentration["HHI"]

    return concentration.round({
        "Total_Weight": 2,
        "Largest_Exposure": 2,
        "HHI": 4,
        "Effective_Categories": 2
    })


sector_concentration = calculate_concentration(sector_data)
country_concentration = calculate_concentration(country_data)

print("Sector concentration:")
display(sector_concentration)

print("\nCountry concentration:")
display(country_concentration)

Sector concentration:


,Total_Weight,Largest_Exposure,Number_of_Categories,HHI,Effective_Categories
Ticker,,,,,
EQQQ.L,100.00,68.51,9,0.5005,2.00
ICHN.AS,100.00,22.36,11,0.1534,6.52
INRG.L,100.00,39.00,4,0.3330,3.00
ISF.L,100.01,28.61,12,0.1559,6.42
IUHC.L,100.00,100.00,1,1.0000,1.00
IUIT.L,100.00,100.00,1,1.0000,1.00
SWDA.L,100.02,30.27,11,0.1589,6.29
VEUR.L,100.00,24.70,12,0.1421,7.04
VFEG.L,100.00,36.60,11,0.2023,4.94



Country concentration:


,Total_Weight,Largest_Exposure,Number_of_Categories,HHI,Effective_Categories
Ticker,,,,,
AGGG.L,100.00,48.33,7,0.3243,3.08
CORP.L,100.00,89.67,3,0.8130,1.23
EQQQ.L,100.00,100.00,1,1.0000,1.00
GHYS.L,100.00,90.67,5,0.8272,1.21
IBTG.L,100.00,99.83,2,0.9966,1.00
ICHN.AS,100.00,100.00,1,1.0000,1.00
IGLO.L,100.00,55.12,8,0.3408,2.93
INRG.L,100.00,100.00,1,1.0000,1.00
ISF.L,100.00,100.00,1,1.0000,1.00


In [12]:
# Save corrected exposure data
sector_data.to_csv(
    "../data/processed/etf_sector_exposure.csv",
    index=False
)

country_data.to_csv(
    "../data/processed/etf_country_exposure.csv",
    index=False
)

# Save concentration metrics
sector_concentration.to_csv(
    "../data/processed/etf_sector_concentration.csv"
)

country_concentration.to_csv(
    "../data/processed/etf_country_concentration.csv"
)

print("Exposure and concentration files saved successfully.")

Exposure and concentration files saved successfully.
